# Ejercicio 6: Parquet vs tablas DuckDB

Este cuaderno muestra la consulta directa a Parquet (6.1), la tabla materializada (6.2), verifica que ambas rutas dan el mismo resultado y analiza los resultados medidos por `scripts/benchmark.py` (`docs/benchmark_resultados.csv`). Para reproducir las mediciones: `docker exec lab8-lab sh -c "python /workspace/scripts/benchmark.py --repeticiones 5"`.

In [1]:
import duckdb, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt, os, glob
pd.set_option('display.width', 200)
BASE = '/workspace'
print('DuckDB', duckdb.__version__)

DuckDB 1.5.5


## 6.1 Consulta directa a Parquet

In [2]:
con = duckdb.connect()
con.sql("""
SELECT count(*) AS viajes, min(filename) AS primer_archivo, max(filename) AS ultimo_archivo
FROM read_parquet(BASE_GLOB, union_by_name = true, filename = true)
""".replace('BASE_GLOB', "'/workspace/data/raw/yellow/*/*.parquet'")).show(max_width=200)

┌──────────┬─────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────┐
│  viajes  │                         primer_archivo                          │                         ultimo_archivo                          │
│  int64   │                             varchar                             │                             varchar                             │
├──────────┼─────────────────────────────────────────────────────────────────┼─────────────────────────────────────────────────────────────────┤
│ 70873075 │ /workspace/data/raw/yellow/2024/yellow_tripdata_2024-01.parquet │ /workspace/data/raw/yellow/2026/yellow_tripdata_2026-08.parquet │
└──────────┴─────────────────────────────────────────────────────────────────┴─────────────────────────────────────────────────────────────────┘



## 6.2 Tabla materializada: tamaño y tiempo de creación

In [3]:
cre = pd.read_csv(f'{BASE}/docs/benchmark_resultados_creacion.csv')
cre['ratio_duckdb_vs_parquet'] = (cre.mb_duckdb / cre.mb_parquet).round(2)
cre.round(2)

,tamano,segundos_creacion,filas,mb_parquet,mb_duckdb,ratio_duckdb_vs_parquet
0,1_mes (2024-01),3.07,3021175,51.32,68.17,1.33
1,3_meses (2024-01..03),4.78,9722363,164.41,212.09,1.29
2,2026 (8 meses),12.27,30040469,519.73,650.92,1.25
3,2024 (12 meses),17.70,41829938,708.92,916.21,1.29
4,2024+2026 (20 meses),23.52,71870407,1228.65,1565.01,1.27


In [4]:
db = f'{BASE}/data/processed/taxi.duckdb'
print('taxi.duckdb: %.0f MB' % (os.path.getsize(db)/1e6))
print('parquet   : %.0f MB' % (sum(os.path.getsize(f) for f in glob.glob(f'{BASE}/data/raw/*/*/*.parquet'))/1e6))
con = duckdb.connect(db, read_only=True)
con.sql('DESCRIBE taxi_trips').show(max_width=200)

taxi.duckdb: 1565 MB
parquet   : 1229 MB
┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ tipo_taxi        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count  │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_distance    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID     │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ payment_type     │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ fare_amount      │ DOUBLE      │ YES     │ NULL    │ NULL

## Verificación de equivalencia (conteos iguales)

In [5]:
pq = """SELECT count(*) FROM read_parquet(['/workspace/data/raw/yellow/*/*.parquet','/workspace/data/raw/green/*/*.parquet'], union_by_name=true)"""
n_pq = duckdb.sql(pq).fetchone()[0]
n_tb = con.sql('SELECT count(*) FROM taxi_trips').fetchone()[0]
print(f'Parquet: {n_pq:,}  Tabla: {n_tb:,}  iguales: {n_pq == n_tb}')
con.sql('SELECT tipo_taxi, anio, count(*) viajes FROM taxi_trips GROUP BY ALL ORDER BY ALL').show()

Parquet: 71,870,407  Tabla: 71,870,407  iguales: True


┌───────────┬───────┬──────────┐
│ tipo_taxi │ anio  │  viajes  │
│  varchar  │ int16 │  int64   │
├───────────┼───────┼──────────┤
│ green     │  2024 │   660218 │
│ green     │  2026 │   337114 │
│ yellow    │  2024 │ 41169720 │
│ yellow    │  2026 │ 29703355 │
└───────────┴───────┴──────────┘



El script `benchmark.py` además compara el resultado completo de cada una de las 7 consultas en los 5 tamaños (35 comparaciones, 0 discrepancias).

In [6]:
con.close()  # un .duckdb admite un solo proceso escritor; cerramos la conexión

## 6.5 y 6.6 Resultados medidos

In [7]:
r = pd.read_csv(f'{BASE}/docs/benchmark_resultados.csv')
orden = list(dict.fromkeys(r.tamano))
print(r.repeticion.max(), 'repeticiones;', len(r), 'mediciones')
frio = r[r.fase=='frio'].pivot_table(index=['tamano','consulta'], columns='fuente', values='segundos')
cal = r[r.fase=='caliente'].groupby(['tamano','consulta','fuente']).segundos.median().unstack()
res = pd.DataFrame({'parquet_frio': frio.parquet, 'tabla_frio': frio.tabla, 'parquet_caliente_mediana': cal.parquet, 'tabla_caliente_mediana': cal.tabla})
res['aceleracion_caliente'] = res.parquet_caliente_mediana / res.tabla_caliente_mediana
res = res.reindex(orden, level=0)
res.round(4)

5 repeticiones; 350 mediciones


parquet_frio  tabla_frio  parquet_caliente_mediana  tabla_caliente_mediana  aceleracion_caliente
tamano                consulta                                                                                                                  
1_mes (2024-01)       Q1_conteo_total                 0.0325      0.0253                    0.0323                  0.0027               11.9078
                      Q2_agregacion_por_mes           0.1165      0.1094                    0.0940                  0.0353                2.6601
                      Q3_promedio_por_tipo            0.1310      0.0838                    0.0812                  0.0146                5.5430
                      Q4_distribucion_por_hora        0.2306      0.1156                    0.1875                  0.0164               11.4013
                      Q5_top_zonas                    0.0815      0.0770                    0.0802                  0.0098                8.2241
                      Q6_percentiles_propina          0.6342      0.4556                    0.3849                  0.3416                1.1267
                      Q7_filtro_selectivo             0.0977      0.0664                    0.0947                  0.0073               13.0498
3_meses (2024-01..03) Q1_conteo_total                 0.0457      0.0192                    0.0441                  0.0030               14.6814
                      Q2_agregacion_por_mes           0.1736      0.2212                    0.1660                  0.0814                2.0398
                      Q3_promedio_por_tipo            0.2285      0.2194                    0.1587                  0.0299                5.3098
                      Q4_distribucion_por_hora        0.3445      0.2713                    0.3527                  0.0420                8.4015
                      Q5_top_zonas                    0.1686      0.1661                    0.1288                  0.0186                6.9222
                      Q6_percentiles_propina          1.4073      1.6376                    1.2355                  1.5178                0.8140
                      Q7_filtro_selectivo             0.1732      0.1250                    0.1682                  0.0164               10.2805
2026 (8 meses)        Q1_conteo_total                 0.0662      0.0217                    0.0760                  0.0045               16.8100
                      Q2_agregacion_por_mes           0.5550      0.8548                    0.4709                  0.2190                2.1497
                      Q3_promedio_por_tipo            0.5947      0.7072                    0.5470                  0.0896                6.1039
                      Q4_distribucion_por_hora        0.9725      0.8935                    0.9633                  0.1093                8.8143
                      Q5_top_zonas                    0.4824      0.5440                    0.4267                  0.0676                6.3107
                      Q6_percentiles_propina          4.0687      4.0901                    4.0076                  3.8341                1.0452
                      Q7_filtro_selectivo             0.5754      0.4424                    0.5096                  0.0533                9.5532
2024 (12 meses)       Q1_conteo_total                 0.1480      0.0265                    0.1890                  0.0062               30.4964
                      Q2_agregacion_por_mes           0.7084      1.0380                    0.6548                  0.2619                2.5005
                      Q3_promedio_por_tipo            0.6386      0.8691                    0.6035                  0.1213                4.9770
                      Q4_distribucion_por_hora        1.3040      1.1275                    1.2166                  0.1405                8.6603
                      Q5_top_zonas                    0.5635      0.6076                    0.4990                  0.0686           

In [8]:
tot = res.groupby(level=0, sort=False).sum()
tot['aceleracion_caliente'] = tot.parquet_caliente_mediana / tot.tabla_caliente_mediana
tot['aceleracion_frio'] = tot.parquet_frio / tot.tabla_frio
tot.round(3)

,parquet_frio,tabla_frio,parquet_caliente_mediana,tabla_caliente_mediana,aceleracion_caliente,aceleracion_frio
tamano,,,,,,
1_mes (2024-01),1.324,0.933,0.955,0.428,2.232,1.419
3_meses (2024-01..03),2.541,2.660,2.254,1.709,1.319,0.956
2026 (8 meses),7.315,7.554,7.001,4.378,1.599,0.968
2024 (12 meses),9.951,9.217,9.122,5.662,1.611,1.080
2024+2026 (20 meses),14.203,14.063,13.733,9.760,1.407,1.010


In [9]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
x = range(len(orden)); w = 0.38
for a, col, tit in [(ax[0], 'caliente', 'Mediana ejecuciones calientes'), (ax[1], 'frio', 'Primera ejecución (fría)')]:
    p = tot['parquet_'+col if col=='frio' else 'parquet_caliente_mediana']; t = tot['tabla_'+col if col=='frio' else 'tabla_caliente_mediana']
    a.bar([i-w/2 for i in x], p, w, label='Parquet directo', color='#4C78A8')
    a.bar([i+w/2 for i in x], t, w, label='Tabla DuckDB', color='#E8833A')
    a.set_xticks(list(x)); a.set_xticklabels([o.split(' (')[0] for o in orden], rotation=20, ha='right')
    a.set_title(tit + ' (suma de las 7 consultas)'); a.set_ylabel('segundos'); a.spines[['top','right']].set_visible(False)
ax[0].legend(frameon=False)
plt.tight_layout(); plt.savefig(f'{BASE}/docs/img/06_benchmark.png', dpi=130); plt.show()

## Amortización del costo de materializar

Cuántas ejecuciones de las 7 consultas (en caliente) hacen falta para que el costo de crear la tabla se compense con el ahorro por ejecución.

In [10]:
am = tot[['parquet_caliente_mediana','tabla_caliente_mediana']].join(cre.set_index('tamano').segundos_creacion)
am['ahorro_por_ronda'] = am.parquet_caliente_mediana - am.tabla_caliente_mediana
am['rondas_para_amortizar'] = am.segundos_creacion / am.ahorro_por_ronda
am.round(2)

,parquet_caliente_mediana,tabla_caliente_mediana,segundos_creacion,ahorro_por_ronda,rondas_para_amortizar
tamano,,,,,
1_mes (2024-01),0.95,0.43,3.07,0.53,5.83
3_meses (2024-01..03),2.25,1.71,4.78,0.54,8.76
2026 (8 meses),7.00,4.38,12.27,2.62,4.68
2024 (12 meses),9.12,5.66,17.70,3.46,5.12
2024+2026 (20 meses),13.73,9.76,23.52,3.97,5.92
